## Loading EfficientNetV2S

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mostafaabla/garbage-classification")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/garbage-classification


In [ ]:
# /kaggle/input/garbage-classification/garbage_classification

In [2]:
import tensorflow as tf

2025-04-29 02:24:32.446438: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745893472.663829      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745893472.721143      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
model_efficient_net = tf.keras.applications.EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
model_efficient_net.trainable = False
print("Done")

I0000 00:00:1745893484.870862      31 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Done


In [ ]:
model_efficient_net.summary()

In [5]:
print(f"Total layers in Partial EfficientNet-50: {len(model_efficient_net.layers)}")

Total layers in Partial EfficientNet-50: 513


## Fine-tune the model

In [6]:
# Build model
model = tf.keras.Sequential([
    model_efficient_net,
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(12, activation='softmax')
])
print("Done")

Done


In [8]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    loss_weights=None,
    metrics=["accuracy"],
    weighted_metrics=None,
    run_eagerly=False,
    steps_per_execution=1,
    jit_compile="auto",
    auto_scale_loss=True,
)

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetv2-s (Functional)        │ (None, 8, 8, 1280)          │      20,331,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 81920)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │      10,485,888 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 12)                  │           1,548 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 30,819,308 (117.57 MB)

 Trainable params: 10,487,692 (40.01 MB)

 Non-trainable params: 20,331,616 (77.56 MB)

## Model Callback

In [10]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [11]:
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

## Reading Directory

In [12]:
import tensorflow as tf
import numpy as np

In [13]:
# Specify the directory containing the images 
# data_directory = 'garbage_classification/' 
data_directory = '/kaggle/input/garbage-classification/garbage_classification'

In [14]:
img_height = 256
img_width = 256
batch_size = 1000

Found 15515 files belonging to 12 classes.
Using 12412 files for training.
Using 3103 files for validation.


In [88]:
# dataset_validation.class_names

['battery',
 'biological',
 'brown-glass',
 'cardboard',
 'clothes',
 'green-glass',
 'metal',
 'paper',
 'plastic',
 'shoes',
 'trash',
 'white-glass']

## Dataset Evaluation

In [22]:
# import os

In [89]:
# dataset_path ='garbage_classification'
# dataset_folder = os.listdir(dataset_path)

In [90]:
# dataset_folder

In [44]:
# labels = []

In [91]:
# for d in dataset_folder:
#     if os.path.isdir(os.path.join(dataset_path, d)):
#         labels.append(d)

In [92]:
# labels

## Data Augumentation

In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [19]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

2025-04-29 07:56:46.435441: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [20]:
train_generator = datagen.flow_from_directory(
    data_directory,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)   

Found 12415 images belonging to 12 classes.


In [21]:
val_generator = datagen.flow_from_directory(
    data_directory,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 3100 images belonging to 12 classes.


In [132]:
# train_generator.class_indices

## Value Mapping

In [54]:
value_mapping = {
    'clothes': 'recyclable',
    'plastic': 'non-recyclable',
    'shoes': 'recyclable',
    'trash': 'non-recyclable',
    'brown-glass': 'non-recyclable',
    'paper': 'non-recyclable',
    'metal': 'non-recyclable',
    'white-glass': 'non-recyclable',
    'battery': 'non-recyclable',
    'biological': 'non-recyclable',
    'green-glass': 'non-recyclable',
    'cardboard': 'non-recyclable'
}
value_mapping.values()

dict_values(['recyclable', 'non-recyclable', 'recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable', 'non-recyclable'])

## Model Training

EfficientNets are currently one of the most powerful convolutional neural network (CNN) models. With the rise of Vision Transformers, which achieved even higher accuracies than EfficientNets, the question arose whether CNNs are now dying. EfficientNetV2 proves this wrong by not just improving accuracies but by also reducing training time and latency.

In [24]:
model.fit(train_generator, epochs=10, validation_data=val_generator, callbacks=[lr_scheduler, early_stopping])

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1745893684.182150      92 service.cc:148] XLA service 0x789be80a0a80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745893684.182763      92 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1745893687.825387      92 cuda_dnn.cc:529] Loaded cuDNN version 90300


  2/388 ━━━━━━━━━━━━━━━━━━━━ 32s 84ms/step - accuracy: 0.1875 - loss: 3.2788   

I0000 00:00:1745893708.248154      92 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


388/388 ━━━━━━━━━━━━━━━━━━━━ 396s 862ms/step - accuracy: 0.7968 - loss: 0.7408 - val_accuracy: 0.8932 - val_loss: 0.3475 - learning_rate: 0.0010
Epoch 2/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 213s 540ms/step - accuracy: 0.9210 - loss: 0.2617 - val_accuracy: 0.9106 - val_loss: 0.2680 - learning_rate: 0.0010
Epoch 3/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 210s 532ms/step - accuracy: 0.9375 - loss: 0.2026 - val_accuracy: 0.9126 - val_loss: 0.2559 - learning_rate: 0.0010
Epoch 4/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 210s 535ms/step - accuracy: 0.9432 - loss: 0.1827 - val_accuracy: 0.9274 - val_loss: 0.2235 - learning_rate: 0.0010
Epoch 5/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 209s 532ms/step - accuracy: 0.9436 - loss: 0.1685 - val_accuracy: 0.9252 - val_loss: 0.2379 - learning_rate: 0.0010
Epoch 6/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 211s 537ms/step - accuracy: 0.9511 - loss: 0.1558 - val_accuracy: 0.9145 - val_loss: 0.2483 - learning_rate: 0.0010
Epoch 7/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 211s 536ms/step - accuracy: 0.9495 

In [26]:
prediction = model.predict(val_generator)

97/97 ━━━━━━━━━━━━━━━━━━━━ 42s 430ms/step


In [27]:
loss, accuracy = model.evaluate(val_generator)

97/97 ━━━━━━━━━━━━━━━━━━━━ 43s 437ms/step - accuracy: 0.9212 - loss: 0.2257


In [39]:
from PIL import Image

image = Image.open('/kaggle/input/bottle/bottle.png')
 
# summarize some details about the image
print(image.format)
print(image.size)
print(image.mode)
np_img = np.array(image)
print(np_img.shape)
print(np_img)

PNG
(256, 256)
P
(256, 256)
[[1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 ...
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]]


In [35]:
prediction = model.predict(np_img)

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("sequential_1/Cast:0", shape=(32, 256), dtype=float32). Expected shape (None, 256, 256, 3), but input has incompatible shape (32, 256)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 256), dtype=uint8)
  • training=False
  • mask=None